In [ ]:
import numpy as np
import pinocchio as pin
from example_robot_data import load

# Init model and data
rob = load("ur10")
model0 = rob.model
data0 = model0.createData()

q = np.array([-0.00159, -1.5216, 1.1084, -1.1216, -0.81159, -0.00159]) # original q matrix
v = np.ones(6) * 0.5
a = np.ones(6) * 1.0

joint_id = model0.getJointId("elbow_joint")

# Recursive Newton Euler to compute torques for the orig model
tau0 = pin.rnea(model0, data0, q, v, a)

Y = np.zeros((6, 10))

def inertia_from_mu(mu):
    m = mu[0]
    h = np.array(mu[1:4])
    
    I = np.array([
        [mu[4], mu[5], mu[6]],
        [mu[5], mu[7], mu[8]],
        [mu[6], mu[8], mu[9]]
    ])

    if abs(m) > 1e-12:
        c = h / m
    else:
        c = np.zeros(3)

    return pin.Inertia(m, c, I) 

for i in range(10): # Iterate throuch the 10 patameters
    mu = np.zeros(10)
    mu[i] = 1.0

    model = model0.copy()
    data = model.createData()

    added_inertia = inertia_from_mu(mu)
    model.inertias[joint_id] = model.inertias[joint_id] + added_inertia

    tau = pin.rnea(model, data, q, v, a)

    Y[:, i] = tau - tau0

print(Y)

[[-1.70987691e-02  0.00000000e+00  0.00000000e+00  0.00000000e+00
   1.20651249e+00 -2.23322323e+00 -5.79691078e-02  0.00000000e+00
  -1.12756442e-01 -2.06512488e-01]
 [ 4.47262597e-02  3.55271368e-15  3.55271368e-15  3.55271368e-15
  -9.19371302e-02 -9.15840579e-01 -1.69381984e-01  2.00000000e+00
   4.01542069e-01  9.19371302e-02]
 [ 3.55271368e-15  3.55271368e-15  3.55271368e-15  3.55271368e-15
  -9.19371302e-02 -9.15840579e-01 -1.69381984e-01  2.00000000e+00
   4.01542069e-01  9.19371302e-02]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00